# 🌍 Machine Learning for Earth Observation
## Unsupervised Classification — Rubavu Secondary City, Rwanda

---

| | |
|---|---|
| **Institution** | University of Rwanda — College of Science and Technology |
| **Department** | Department of Spatial Planning |
| **Programme** | URP/EPM — Year 3 |
| **Module** | SP80663: Machine Learning for Earth Observation |
| **Lecturer** | Mr. Hyacinthe NGWIJABAGABO |
| **Study Area** | Rubavu Secondary City, Rwanda |

---

## 📖 Learning Objectives

By the end of this tutorial, students will be able to:
1. Authenticate and initialize Google Earth Engine (GEE) in Python
2. Define an Area of Interest (AOI) using polygon coordinates
3. Load and preprocess Sentinel-2 satellite imagery
4. Apply **K-Means clustering** (unsupervised classification)
5. Visualize and interpret classification results
6. Export results for further analysis in GIS software

---

## 🗺️ Study Area: Rubavu

Rubavu is one of Rwanda's secondary cities located in the **Western Province**, bordering the Democratic Republic of Congo along Lake Kivu. It is known for its urban growth, tourism, and strategic geographic position. This tutorial uses Sentinel-2 imagery to classify land cover types within Rubavu.

---

> **⚠️ Pre-requisite:** You must have a Google Earth Engine account registered. If not, register at: https://signup.earthengine.google.com/

---
## 📦 STEP 1 — Install & Import Required Libraries

We begin by installing the **Earth Engine Python API** (`earthengine-api`) and `geemap`, a powerful mapping package built on top of GEE for interactive visualization.

In [ ]:
# ============================================================
# STEP 1: Install required packages
# Run this cell first — it may take 1-2 minutes
# ============================================================

!pip install earthengine-api geemap --quiet

print("✅ Packages installed successfully!")

In [ ]:
# ============================================================
# STEP 1b: Import all necessary Python libraries
# ============================================================

import ee                    # Google Earth Engine API
import geemap                # Interactive mapping library
import numpy as np           # Numerical operations
import matplotlib.pyplot as plt   # Plotting
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

print("✅ All libraries imported successfully!")
print(f"   - earthengine-api version: {ee.__version__}")
print(f"   - geemap version: {geemap.__version__}")

---
## 🔐 STEP 2 — Authenticate & Initialize Google Earth Engine

Authentication links your Google account to the Earth Engine API. 

- **First time:** A browser popup will ask you to log in and authorize the application.
- **After authentication:** The session is saved and future runs only need `ee.Initialize()`.

> 💡 **Tip:** If you get an error, try running the cell again or restart the Colab runtime.

In [ ]:
# ============================================================
# STEP 2: Authenticate and Initialize GEE
# Your registered GEE project: ee-ngwijabagabohyacinthe
# ============================================================

# Authenticate (opens a browser popup on first run)
try:
    ee.Initialize(project='ee-ngwijabagabohyacinthe')
    print("✅ Earth Engine initialized successfully!")
    print("   Project: ee-ngwijabagabohyacinthe")
except Exception as e:
    print("⚠️  Authentication required. Running ee.Authenticate()...")
    ee.Authenticate()
    ee.Initialize(project='ee-ngwijabagabohyacinthe')
    print("✅ Earth Engine authenticated and initialized!")

---
## 📌 STEP 3 — Define the Area of Interest (AOI): Rubavu

We define Rubavu's boundary using a **polygon** with the coordinates provided. In GEE, geometries are defined using `ee.Geometry.Polygon()` with **[longitude, latitude]** pairs.

> **📐 Coordinate System:** GEE uses **WGS84 (EPSG:4326)** by default.

In [ ]:
# ============================================================
# STEP 3: Define the Area of Interest — Rubavu Secondary City
# Coordinates format: [longitude, latitude]
# ============================================================

# Rubavu AOI polygon vertices (closing the polygon by repeating vertex 0)
rubavu_coords = [
    [29.270096034902416, -1.6920931654092402],   # Vertex 0 (start)
    [29.25052663792976,  -1.6822269152867393],   # Vertex 1
    [29.24503347386726,  -1.6977555123959842],   # Vertex 2
    [29.253616542714916, -1.7079648536488905],   # Vertex 3
    [29.262371272939525, -1.705734245894214],    # Vertex 4
    [29.270096034902416, -1.6920931654092402],   # Vertex 5 (closing = Vertex 0)
]

# Create GEE Geometry object
aoi = ee.Geometry.Polygon(rubavu_coords)

# Calculate area
area_m2 = aoi.area().getInfo()
area_km2 = area_m2 / 1e6

print("✅ AOI defined successfully!")
print(f"   Location  : Rubavu Secondary City, Rwanda")
print(f"   Vertices  : {len(rubavu_coords)} points")
print(f"   Area      : {area_km2:.2f} km²")
print(f"   Centroid  : ~{-1.697:.4f}°N, {29.257:.4f}°E")

---
## 🗺️ STEP 4 — Visualize the AOI on an Interactive Map

Before loading satellite data, let's confirm our AOI is correctly positioned over Rubavu using `geemap`.

In [ ]:
# ============================================================
# STEP 4: Display the AOI on an interactive map
# ============================================================

# Create interactive map centered on Rubavu
Map = geemap.Map()
Map.centerObject(aoi, zoom=13)

# Add AOI outline in red
Map.addLayer(
    aoi, 
    {'color': 'red', 'fillColor': '00000000', 'width': 2}, 
    'Rubavu AOI'
)

# Switch to satellite basemap
Map.add_basemap('SATELLITE')

print("✅ Map created! The red polygon outlines Rubavu's AOI.")
Map

---
## 🛰️ STEP 5 — Load Sentinel-2 Satellite Imagery

**Sentinel-2** is a European Space Agency (ESA) satellite that provides:
- **Spatial resolution:** 10m (Bands 2,3,4,8), 20m (Bands 5,6,7,8A,11,12), 60m (Bands 1,9,10)
- **Temporal resolution:** ~5-day revisit
- **Free and open access** via GEE

We will:
1. Filter the image collection by **date** and **location**
2. Filter for **low cloud cover** (< 20%)
3. Create a **median composite** to reduce cloud contamination
4. Scale the reflectance values

In [ ]:
# ============================================================
# STEP 5: Load and preprocess Sentinel-2 imagery
# ============================================================

# --- 5a. Define time period ---
START_DATE = '2023-05-01'
END_DATE   = '2023-09-30'
CLOUD_THRESHOLD = 20  # Max cloud cover percentage

print(f"📅 Time period: {START_DATE} → {END_DATE}")
print(f"☁️  Max cloud cover: {CLOUD_THRESHOLD}%")

# --- 5b. Load Sentinel-2 Surface Reflectance collection ---
s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)                               # Spatial filter: only images overlapping AOI
    .filterDate(START_DATE, END_DATE)                # Temporal filter
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))  # Cloud filter
)

# --- 5c. Check number of available images ---
n_images = s2_collection.size().getInfo()
print(f"\n✅ Sentinel-2 collection loaded!")
print(f"   Available images : {n_images}")

# --- 5d. Cloud masking function ---
def mask_s2_clouds(image):
    """Mask clouds and cloud shadows using the SCL band."""
    scl = image.select('SCL')  # Scene Classification Layer
    # Keep only: vegetation(4), non-veg(5), water(6), unclassified(7), snow(11)
    cloud_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(cloud_mask)

# --- 5e. Apply cloud mask and create median composite ---
s2_masked = s2_collection.map(mask_s2_clouds)

s2_image = (
    s2_masked
    .median()                   # Take median value across time (reduces clouds)
    .clip(aoi)                  # Clip to AOI extent
    .divide(10000)              # Scale reflectance: DN → [0, 1]
)

print("\n✅ Median composite created and clipped to AOI!")
print("   Reflectance values scaled to [0, 1]")

---
## 🌈 STEP 6 — Visualize the Sentinel-2 Composite

We'll display the image in:
- **True Color (RGB):** Bands 4-3-2 → Natural appearance
- **False Color (NIR-R-G):** Bands 8-4-3 → Vegetation appears bright red

In [ ]:
# ============================================================
# STEP 6: Visualize the Sentinel-2 composite
# ============================================================

# Visualization parameters
vis_true_color = {
    'bands': ['B4', 'B3', 'B2'],      # Red, Green, Blue
    'min': 0, 'max': 0.3,
    'gamma': 1.4
}

vis_false_color = {
    'bands': ['B8', 'B4', 'B3'],      # NIR, Red, Green
    'min': 0, 'max': 0.5,
    'gamma': 1.4
}

# Add layers to the map
Map2 = geemap.Map()
Map2.centerObject(aoi, zoom=14)
Map2.add_basemap('SATELLITE')

Map2.addLayer(s2_image, vis_true_color,  'True Color (RGB: 4-3-2)')
Map2.addLayer(s2_image, vis_false_color, 'False Color (NIR-R-G: 8-4-3)', shown=False)
Map2.addLayer(aoi, {'color': 'yellow', 'fillColor': '00000000'}, 'AOI Boundary')

print("✅ Sentinel-2 imagery layers added!")
print("   Toggle layers using the layer panel (top right of map)")
print("   - True Color  : Natural colors (useful for urban/water identification)")
print("   - False Color : NIR composite (vegetation = bright red)")
Map2

---
## 🔢 STEP 7 — Compute Spectral Indices

Spectral indices enhance specific land cover types and improve classification accuracy:

| Index | Formula | Highlights |
|-------|---------|------------|
| **NDVI** | (NIR - Red) / (NIR + Red) | Vegetation (dense green = +1) |
| **NDWI** | (Green - NIR) / (Green + NIR) | Water bodies (water = positive) |
| **NDBI** | (SWIR - NIR) / (SWIR + NIR) | Built-up/urban areas |
| **SAVI** | 1.5 × (NIR - Red) / (NIR + Red + 0.5) | Vegetation (accounts for soil) |

In [ ]:
# ============================================================
# STEP 7: Calculate spectral indices
# ============================================================

# Extract required bands
B2  = s2_image.select('B2')   # Blue
B3  = s2_image.select('B3')   # Green
B4  = s2_image.select('B4')   # Red
B8  = s2_image.select('B8')   # NIR
B11 = s2_image.select('B11')  # SWIR-1
B12 = s2_image.select('B12')  # SWIR-2

# --- NDVI: Normalized Difference Vegetation Index ---
ndvi = B8.subtract(B4).divide(B8.add(B4)).rename('NDVI')

# --- NDWI: Normalized Difference Water Index ---
ndwi = B3.subtract(B8).divide(B3.add(B8)).rename('NDWI')

# --- NDBI: Normalized Difference Built-up Index ---
ndbi = B11.subtract(B8).divide(B11.add(B8)).rename('NDBI')

# --- SAVI: Soil Adjusted Vegetation Index ---
L = 0.5  # Soil brightness correction factor
savi = B8.subtract(B4).multiply(1 + L).divide(B8.add(B4).add(L)).rename('SAVI')

print("✅ Spectral indices calculated:")
print("   NDVI  → Vegetation index (range: -1 to +1)")
print("   NDWI  → Water index     (range: -1 to +1)")
print("   NDBI  → Built-up index  (range: -1 to +1)")
print("   SAVI  → Soil-adj. veg.  (range: -1 to +1.5)")

---
## 🧩 STEP 8 — Stack Bands for Classification

We combine the original Sentinel-2 bands with our computed indices into a **multi-band image stack**. This is the input feature space for the K-Means algorithm.

> **Feature Engineering:** Including spectral indices alongside raw bands gives the classifier more discriminative power to separate land cover classes.

In [ ]:
# ============================================================
# STEP 8: Create multi-band feature stack for classification
# ============================================================

# Select core optical bands from Sentinel-2
s2_bands = s2_image.select(['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12'])

# Stack: optical bands + spectral indices
feature_stack = s2_bands.addBands([ndvi, ndwi, ndbi, savi])

# Get list of all band names
band_names = feature_stack.bandNames().getInfo()

print("✅ Feature stack created!")
print(f"   Total features (bands): {len(band_names)}")
print(f"   Band names: {band_names}")
print("\n   Feature breakdown:")
print("   • Sentinel-2 bands: B2 (Blue), B3 (Green), B4 (Red),")
print("                       B5-B7 (Red-Edge), B8 (NIR), B8A (Narrow NIR),")
print("                       B11-B12 (SWIR)")
print("   • Spectral indices: NDVI, NDWI, NDBI, SAVI")

---
## 🤖 STEP 9 — Unsupervised K-Means Classification

### What is K-Means Clustering?

**K-Means** is an unsupervised machine learning algorithm that groups pixels into `k` clusters based on spectral similarity — **without using any labeled training data**.

**Algorithm steps:**
1. Randomly initialize `k` cluster centroids
2. Assign each pixel to the nearest centroid (Euclidean distance)
3. Recalculate centroids as the mean of all assigned pixels
4. Repeat steps 2-3 until convergence

**In GEE:**
- `ee.Clusterer.wekaKMeans(k)` — uses the Weka implementation
- We sample pixels from the image, train the clusterer, then apply it

> **🎯 Choosing k (number of clusters):**
> - Too few → classes are merged (e.g., forest + crops = one class)
> - Too many → classes are split (e.g., dense forest and sparse forest = two classes)
> - Common practice: start with k=8–15 and group similar classes afterward

In [ ]:
# ============================================================
# STEP 9: K-Means Unsupervised Classification
# ============================================================

# --- 9a. Set clustering parameters ---
N_CLUSTERS   = 10    # Number of clusters (classes)
N_SAMPLES    = 5000  # Number of random pixels to train on
RANDOM_SEED  = 42    # For reproducibility
SCALE        = 10    # Spatial resolution in meters (Sentinel-2 native: 10m)

print(f"⚙️  K-Means Parameters:")
print(f"   Number of clusters : {N_CLUSTERS}")
print(f"   Training samples   : {N_SAMPLES} pixels")
print(f"   Random seed        : {RANDOM_SEED}")
print(f"   Scale              : {SCALE} m")

# --- 9b. Sample pixels for training ---
print("\n⏳ Sampling pixels from feature stack...")
training_samples = feature_stack.sample(
    region   = aoi,
    scale    = SCALE,
    numPixels= N_SAMPLES,
    seed     = RANDOM_SEED,
    geometries = False
)

# --- 9c. Train (fit) the K-Means clusterer ---
print("⏳ Training K-Means clusterer...")
clusterer = ee.Clusterer.wekaKMeans(
    nClusters = N_CLUSTERS,
    seed      = RANDOM_SEED
).train(training_samples)

# --- 9d. Apply clusterer to the full image ---
print("⏳ Applying clusterer to the full AOI...")
classified = feature_stack.cluster(clusterer).clip(aoi)

print("\n✅ K-Means classification complete!")
print(f"   Output: Single-band image with {N_CLUSTERS} cluster labels (0 to {N_CLUSTERS-1})")

---
## 🎨 STEP 10 — Visualize Classification Results

We assign a distinct color to each cluster for visualization. The cluster numbers (0-9) are **arbitrary** — we will interpret their meaning in the next step.

In [ ]:
# ============================================================
# STEP 10: Visualize classification results
# ============================================================

# Define 10 distinct colors for the clusters
cluster_colors = [
    '#1a9641',  # Cluster 0 — dark green
    '#52b788',  # Cluster 1 — medium green
    '#b7e4c7',  # Cluster 2 — light green
    '#ffffbe',  # Cluster 3 — pale yellow
    '#d9a021',  # Cluster 4 — golden
    '#c1440e',  # Cluster 5 — reddish brown
    '#4292c6',  # Cluster 6 — water blue
    '#08519c',  # Cluster 7 — deep blue
    '#d4d4d4',  # Cluster 8 — light grey (bare/urban)
    '#636363',  # Cluster 9 — dark grey (built-up)
]

# GEE visualization parameters using palette
vis_classified = {
    'min': 0,
    'max': N_CLUSTERS - 1,
    'palette': cluster_colors
}

# Create new map
Map3 = geemap.Map()
Map3.centerObject(aoi, zoom=14)
Map3.add_basemap('HYBRID')

# Add layers
Map3.addLayer(s2_image,  vis_true_color,  'True Color (Reference)', shown=False)
Map3.addLayer(classified, vis_classified, f'K-Means ({N_CLUSTERS} clusters)')
Map3.addLayer(aoi, {'color': 'white', 'fillColor': '00000000'}, 'AOI Boundary')

print("✅ Classification map displayed!")
print("   Each color = one spectral cluster")
print("   Toggle True Color layer to compare with satellite imagery")
Map3

---
## 📊 STEP 11 — Class Interpretation & Legend

After visual inspection of the clusters against the true color image and spectral index values, we assign **land cover labels** to each cluster.

> **🔬 Interpretation Method:**
> 1. Compare each cluster color with the True Color RGB image
> 2. Check NDVI values (high NDVI = vegetation)
> 3. Check NDWI values (high NDWI = water)
> 4. Check NDBI values (high NDBI = urban/built-up)
> 5. Use Google Maps / local knowledge to verify

> **⚠️ Note:** Labels below are tentative assignments. Students should verify these by overlaying with Google satellite imagery!

In [ ]:
# ============================================================
# STEP 11: Class labeling and legend
# Modify these labels based on YOUR visual interpretation!
# ============================================================

# Assign tentative labels to clusters
# TODO: Students — modify these labels after visual inspection!
class_labels = {
    0: ('Dense Vegetation / Forest',   '#1a9641'),
    1: ('Shrubland / Bush',            '#52b788'),
    2: ('Cropland / Agricultural',     '#b7e4c7'),
    3: ('Sparse Vegetation / Grassland','#ffffbe'),
    4: ('Bare Soil / Exposed Land',    '#d9a021'),
    5: ('Mixed Land / Transition',     '#c1440e'),
    6: ('Water Body (Lake Kivu)',      '#4292c6'),
    7: ('Deep Water / Wetland',        '#08519c'),
    8: ('Low-density Built-up',        '#d4d4d4'),
    9: ('High-density Urban',          '#636363'),
}

# Create a matplotlib legend figure
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis('off')

# Title
fig.suptitle(
    'K-Means Classification — Rubavu Secondary City\n'
    'SP80663: Machine Learning for Earth Observation | Univ. of Rwanda',
    fontsize=12, fontweight='bold', y=0.98
)

# Legend patches
legend_elements = [
    Patch(facecolor=color, edgecolor='black', linewidth=0.5,
          label=f'Cluster {k}: {label}')
    for k, (label, color) in class_labels.items()
]

ax.legend(
    handles=legend_elements,
    loc='center',
    ncol=2,
    fontsize=10,
    frameon=True,
    edgecolor='grey',
    title='Land Cover Classes',
    title_fontsize=11
)

plt.tight_layout()
plt.savefig('rubavu_classification_legend.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Legend displayed and saved as 'rubavu_classification_legend.png'")

---
## 📐 STEP 12 — Calculate Class Areas

We compute the **pixel count** (and hence area in km²) for each cluster using GEE's `reduceRegion` function with `ee.Reducer.frequencyHistogram()`.

In [ ]:
# ============================================================
# STEP 12: Calculate area statistics per class
# ============================================================

print("⏳ Computing class areas (this may take ~30 seconds)...\n")

# Count pixels per cluster using frequency histogram
hist = classified.reduceRegion(
    reducer  = ee.Reducer.frequencyHistogram(),
    geometry = aoi,
    scale    = SCALE,
    maxPixels= 1e9
).getInfo()

# Extract histogram data
pixel_counts = hist.get('cluster', {})

# Pixel area at 10m resolution
pixel_area_m2  = SCALE * SCALE   # 100 m²
pixel_area_km2 = pixel_area_m2 / 1e6

# Build results table
print("-" * 65)
print(f"{'Cluster':^8} {'Land Cover Class':^32} {'Pixels':^10} {'Area (km²)':^10}")
print("-" * 65)

total_pixels = 0
areas = {}

for cluster_id in range(N_CLUSTERS):
    label, _ = class_labels[cluster_id]
    count = int(pixel_counts.get(str(cluster_id), 0))
    area_km2 = count * pixel_area_km2
    areas[cluster_id] = area_km2
    total_pixels += count
    print(f"  {cluster_id:^6}   {label:<32} {count:>8,}   {area_km2:>8.3f}")

total_area = total_pixels * pixel_area_km2
print("-" * 65)
print(f"  {'TOTAL':<40} {total_pixels:>8,}   {total_area:>8.3f}")
print("-" * 65)
print(f"\n✅ Total classified area: {total_area:.3f} km²")

In [ ]:
# ============================================================
# STEP 12b: Visualize class areas as a bar chart
# ============================================================

labels_list  = [class_labels[i][0] for i in range(N_CLUSTERS)]
colors_list  = [class_labels[i][1] for i in range(N_CLUSTERS)]
area_list    = [areas.get(i, 0)    for i in range(N_CLUSTERS)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    'Land Cover Area Statistics — Rubavu, Rwanda\nSP80663: Machine Learning for Earth Observation',
    fontsize=13, fontweight='bold'
)

# Bar chart
bars = ax1.barh(
    [f"C{i}: {l[:25]}" for i, l in enumerate(labels_list)],
    area_list,
    color=colors_list,
    edgecolor='black',
    linewidth=0.6
)
ax1.set_xlabel('Area (km²)', fontsize=11)
ax1.set_title('Area by Land Cover Class', fontsize=12)
ax1.invert_yaxis()
# Add value labels
for bar, val in zip(bars, area_list):
    ax1.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=8)

# Pie chart
wedges, texts, autotexts = ax2.pie(
    area_list,
    labels=[f'C{i}' for i in range(N_CLUSTERS)],
    colors=colors_list,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1}
)
for text in autotexts:
    text.set_fontsize(8)
ax2.set_title('Proportional Land Cover', fontsize=12)

plt.tight_layout()
plt.savefig('rubavu_area_statistics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Charts saved as 'rubavu_area_statistics.png'")

---
## 🔄 STEP 13 — Experiment: Try Different Numbers of Clusters

A key parameter in K-Means is **k** (number of clusters). Let's compare results for different values of k to understand its effect on classification.

In [ ]:
# ============================================================
# STEP 13: Compare k=5, k=10, k=15 classifications
# ============================================================

k_values = [5, 10, 15]
classified_maps = {}

# Reusable palette for up to 15 classes
full_palette = [
    '#1a9641','#52b788','#b7e4c7','#ffffbe','#d9a021',
    '#c1440e','#4292c6','#08519c','#d4d4d4','#636363',
    '#ffd700','#ff6347','#9370db','#20b2aa','#ff69b4'
]

# --- Multi-map comparison ---
Map4 = geemap.Map()
Map4.centerObject(aoi, zoom=13)

for k in k_values:
    print(f"⏳ Training K-Means with k={k}...")
    
    clusterer_k = ee.Clusterer.wekaKMeans(
        nClusters=k, seed=RANDOM_SEED
    ).train(training_samples)
    
    classified_k = feature_stack.cluster(clusterer_k).clip(aoi)
    classified_maps[k] = classified_k
    
    vis_k = {'min': 0, 'max': k-1, 'palette': full_palette[:k]}
    Map4.addLayer(classified_k, vis_k, f'K-Means k={k}', shown=(k==10))
    print(f"   ✅ k={k} complete")

Map4.addLayer(aoi, {'color': 'white', 'fillColor': '00000000'}, 'AOI Boundary')

print("\n✅ All three classifications added to map!")
print("   Toggle between k=5, k=10, k=15 using the layer panel")
print("   🔍 Observe: How do the boundaries change as k increases?")
Map4

---
## 💾 STEP 14 — Export Results to Google Drive

We export the classified image as a **GeoTIFF** file to Google Drive so it can be opened in QGIS, ArcGIS, or other GIS software for further analysis.

In [ ]:
# ============================================================
# STEP 14: Export classification to Google Drive
# ============================================================

# Export the 10-cluster classification
export_task = ee.batch.Export.image.toDrive(
    image        = classified.toInt8(),         # Convert to integer (saves space)
    description  = 'Rubavu_KMeans_10clusters',  # Task name in GEE
    folder       = 'GEE_SP80663',               # Google Drive folder
    fileNamePrefix = 'Rubavu_Unsupervised_KMeans10',
    region       = aoi,
    scale        = 10,                          # 10m resolution
    crs          = 'EPSG:4326',                 # WGS84
    maxPixels    = 1e9,
    fileFormat   = 'GeoTIFF'
)

# Start the export task
export_task.start()

print("✅ Export task submitted to Google Earth Engine!")
print(f"   Task name  : Rubavu_KMeans_10clusters")
print(f"   Drive folder: GEE_SP80663")
print(f"   File name  : Rubavu_Unsupervised_KMeans10.tif")
print(f"   Resolution : 10 m")
print(f"   Format     : GeoTIFF (EPSG:4326)")
print("\n⏳ Export is running in the background...")
print("   Monitor progress at: https://code.earthengine.google.com/tasks")
print("   File will appear in your Google Drive under 'GEE_SP80663' folder")

In [ ]:
# ============================================================
# STEP 14b: Also export the Sentinel-2 composite (optional)
# ============================================================

# Export True Color composite for reference
export_s2 = ee.batch.Export.image.toDrive(
    image        = s2_image.select(['B4','B3','B2']).multiply(10000).toInt16(),
    description  = 'Rubavu_Sentinel2_TrueColor',
    folder       = 'GEE_SP80663',
    fileNamePrefix = 'Rubavu_S2_TrueColor',
    region       = aoi,
    scale        = 10,
    crs          = 'EPSG:4326',
    maxPixels    = 1e9,
    fileFormat   = 'GeoTIFF'
)
export_s2.start()

print("✅ Sentinel-2 True Color composite export submitted!")
print("   File: Rubavu_S2_TrueColor.tif")

---
## 🔬 STEP 15 — Spectral Profile Analysis

We compute the **mean spectral values** for each cluster to understand their spectral signatures. This helps confirm our class label interpretations.

In [ ]:
# ============================================================
# STEP 15: Spectral profile analysis per cluster
# ============================================================

print("⏳ Computing mean spectral indices per cluster...\n")

# Add cluster band to the spectral indices image
indices_with_cluster = ee.Image.cat([
    ndvi.rename('NDVI'),
    ndwi.rename('NDWI'),
    ndbi.rename('NDBI'),
    savi.rename('SAVI'),
    classified.rename('cluster')
])

# For each cluster, sample mean index values
cluster_stats = []

for k in range(N_CLUSTERS):
    # Mask to just this cluster
    cluster_mask = classified.eq(k)
    
    stats = ee.Image.cat([ndvi, ndwi, ndbi, savi]).updateMask(cluster_mask).reduceRegion(
        reducer  = ee.Reducer.mean(),
        geometry = aoi,
        scale    = 30,
        maxPixels= 1e8
    ).getInfo()
    
    cluster_stats.append(stats)

# Display results
print(f"{'Cluster':^8} {'NDVI':^8} {'NDWI':^8} {'NDBI':^8} {'SAVI':^8} {'Interpretation':^30}")
print("-" * 75)

for k, stats in enumerate(cluster_stats):
    ndvi_v = stats.get('NDVI', float('nan'))
    ndwi_v = stats.get('NDWI', float('nan'))
    ndbi_v = stats.get('NDBI', float('nan'))
    savi_v = stats.get('SAVI', float('nan'))
    label  = class_labels[k][0][:28]
    
    print(f"  C{k:^5}  {ndvi_v:>6.3f}  {ndwi_v:>6.3f}  {ndbi_v:>6.3f}  {savi_v:>6.3f}  {label:<28}")

print("-" * 75)
print("\n📊 Interpretation guide:")
print("   NDVI > 0.4  → Dense vegetation")
print("   NDWI > 0.2  → Water body")
print("   NDBI > 0.0  → Urban / built-up")
print("   NDVI ≈ 0.1-0.3 → Sparse veg / cropland")

In [ ]:
# ============================================================
# STEP 15b: Plot spectral profiles as grouped bar chart
# ============================================================

import numpy as np

indices_labels = ['NDVI', 'NDWI', 'NDBI', 'SAVI']
index_keys     = ['NDVI', 'NDWI', 'NDBI', 'SAVI']

x = np.arange(N_CLUSTERS)
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))

index_colors = ['#2ca25f', '#2166ac', '#d73027', '#8856a7']

for i, (idx_key, idx_label, idx_color) in enumerate(zip(index_keys, indices_labels, index_colors)):
    values = [cluster_stats[k].get(idx_key, 0) or 0 for k in range(N_CLUSTERS)]
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, values, width, label=idx_label, 
                  color=idx_color, alpha=0.85, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Cluster Number', fontsize=11)
ax.set_ylabel('Mean Index Value', fontsize=11)
ax.set_title(
    'Mean Spectral Index Values per K-Means Cluster\n'
    'Rubavu, Rwanda | SP80663: Machine Learning for Earth Observation',
    fontsize=12, fontweight='bold'
)
ax.set_xticks(x)
ax.set_xticklabels([f'C{i}\n{class_labels[i][0][:15]}' for i in range(N_CLUSTERS)], fontsize=8)
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.legend(title='Spectral Indices', fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('rubavu_spectral_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Spectral profile chart saved as 'rubavu_spectral_profiles.png'")

---
## 📝 STEP 16 — Summary & Student Exercises

### 📋 What We Accomplished

| Step | Action | Output |
|------|--------|--------|
| 1-2  | Install libraries & authenticate GEE | GEE session ready |
| 3-4  | Define and visualize AOI (Rubavu) | Polygon map |
| 5-6  | Load Sentinel-2 median composite | Cloud-free image |
| 7-8  | Compute spectral indices & feature stack | 14-band input |
| 9-10 | K-Means clustering (k=10) | Classified map |
| 11   | Class labeling & legend | Land cover map |
| 12   | Area statistics | Class areas (km²) |
| 13   | Sensitivity analysis (k=5,10,15) | Comparison maps |
| 14   | Export to Google Drive | GeoTIFF files |
| 15   | Spectral profile analysis | Index charts |

---

### 🎯 Student Exercises

Complete the following exercises and include your answers in your lab report:

**Exercise 1 — Parameter Sensitivity**
- Run the classification with `k = 6` and `k = 12`. Compare the results.
- Which value of k gives the most meaningful land cover classes for Rubavu? Justify your answer.

**Exercise 2 — Class Interpretation**
- Open the exported GeoTIFF in QGIS and overlay with Google Satellite imagery.
- Revise the class labels based on your visual inspection.
- Identify at least one class that represents **Lake Kivu** and one that represents **urban Rubavu**.

**Exercise 3 — Spectral Analysis**
- Using the spectral profiles chart from Step 15:
  - Which cluster has the highest NDVI? What does this suggest?
  - Which cluster has the highest NDWI? Is this consistent with its geographic location?
  - Which cluster has the highest NDBI? What land cover does it represent?

**Exercise 4 — Time Series (Advanced)**
- Repeat the classification using imagery from a different season: `'2023-01-01'` to `'2023-04-30'`
- Compare the two results. Are there any significant changes in land cover?

**Exercise 5 — Critical Evaluation**
- What are THREE limitations of unsupervised classification compared to supervised classification?
- How could you improve the classification accuracy without using training samples?

---

### 📚 Key Concepts Summary

| Concept | Definition |
|---------|------------|
| **Unsupervised Classification** | Grouping pixels by spectral similarity without prior knowledge or training labels |
| **K-Means** | Iterative algorithm that partitions data into k clusters by minimizing within-cluster variance |
| **Spectral Index** | Mathematical transformation of band values to highlight specific surface features |
| **Median Composite** | Image created from the median value per pixel over time, reducing cloud effects |
| **Feature Space** | Multi-dimensional space defined by all input band values |
| **Centroid** | The mean position of all pixels in a cluster |

---

### 🔗 Useful Resources
- [GEE Clusterer Documentation](https://developers.google.com/earth-engine/apidocs/ee-clusterer-wekakmeans)
- [Sentinel-2 Band Description](https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2/instrument-payload)
- [geemap Documentation](https://geemap.org/)
- [GEE Code Editor](https://code.earthengine.google.com/)

---

*Prepared by: Mr. Hyacinthe NGWIJABAGABO*  
*University of Rwanda | College of Science and Technology*  
*Department of Spatial Planning | SP80663: Machine Learning for Earth Observation*

In [ ]:
# ============================================================
# FINAL CELL: Complete summary printout
# ============================================================

print("=" * 60)
print("   SP80663: Machine Learning for Earth Observation")
print("   Unsupervised Classification — Rubavu, Rwanda")
print("   University of Rwanda | Dept. of Spatial Planning")
print("=" * 60)
print()
print("📍 AOI       : Rubavu Secondary City")
print("🛰️  Sensor    : Sentinel-2 (10m resolution)")
print("📅 Period    : 2023-05-01 to 2023-09-30")
print(f"🤖 Method    : K-Means Clustering (k={N_CLUSTERS})")
print(f"📊 Features  : 14 bands (10 S2 + 4 indices)")
print(f"🏷️  Classes   : {N_CLUSTERS} spectral clusters")
print()
print("📁 Outputs:")
print("   • Rubavu_Unsupervised_KMeans10.tif  → Google Drive")
print("   • Rubavu_S2_TrueColor.tif           → Google Drive")
print("   • rubavu_classification_legend.png  → Colab files")
print("   • rubavu_area_statistics.png        → Colab files")
print("   • rubavu_spectral_profiles.png      → Colab files")
print()
print("✅ Tutorial complete! Good luck with your exercises.")
print("=" * 60)